In [179]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'darkgrey'

In [180]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.sel(time=slice('1992-01-01', '2024-12-31')).to_dataframe().reset_index()
sst_df = sst_df.query('time_bnds > 0 and nbnds == 0')
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year

In [181]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [182]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst['std'] = sst_df.dropna().groupby(['lat', 'lon', 'month']).std().reset_index()['sst']
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [183]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']
sst_anomaly['normalized_sst_anomaly'] = sst_anomaly['sst_anomaly'] / sst_anomaly['std']

In [184]:
chirps_eastern_east_africa = chirps.sel(time=slice('1993-01-01', '2024-12-01'), latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',   # March, April, May
}

chirps_eastern_east_africa['season'] = chirps_eastern_east_africa['month'].map(month_to_season)

chirps_eastern_east_africa = chirps_eastern_east_africa.dropna(subset=['season'])

chirps_eastern_east_africa_monthly = chirps_eastern_east_africa.groupby(['year', 'month'])['precip'].mean().reset_index()

chirps_eastern_east_africa_season = chirps_eastern_east_africa.groupby(['year', 'season'])[['precip']].mean().reset_index()

In [185]:
def get_tercile_labels(chirps):
    tercile_list = chirps.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()
    bn_list = chirps.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    n_list = chirps.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    an_list = chirps.query(f'{tercile_list[1]} <= precip')['year'].to_list()
    season_dict = {'an': an_list, 'bn': bn_list, 'n': n_list}

    year_to_category_map = {}
    for category, year_list in season_dict.items():
        for year in year_list:
            year_to_category_map[year] = category

    chirps['tercile'] = chirps['year'].map(year_to_category_map)

    return chirps

In [186]:
dict = {}
for i, df in chirps_eastern_east_africa_monthly.groupby(['month']):
    tercile_list = df.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()
    bn_list = df.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    n_list = df.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    an_list = df.query(f'{tercile_list[1]} <= precip')['year'].to_list()
    season_dict = {'an': an_list, 'bn': bn_list, 'n': n_list}
    year_to_category_map = {}
    for category, year_list in season_dict.items():
        for year in year_list:
            year_to_category_map[year] = category

    df['tercile'] = df['year'].map(year_to_category_map)
    dict[i] = df

In [187]:
labeled_chirps_monthly = pd.concat(dict.values()).reset_index().drop('index', axis = 1)

In [203]:
labeled_chirps_seasonal = get_tercile_labels(chirps_eastern_east_africa_season)
labeled_chirps_monthly_seasonal_tercile = chirps_eastern_east_africa_monthly.merge(labeled_chirps_seasonal[['year', 'tercile']], how='left', on='year')

In [202]:
labeled_chirps_monthly

,year,month,precip,tercile
0,1993,3,12.088976,bn
1,1994,3,16.170179,bn
2,1995,3,39.480980,an
3,1996,3,36.686726,an
4,1997,3,36.074913,an
...,...,...,...,...
91,2020,5,56.367199,bn
92,2021,5,70.332794,n
93,2022,5,46.031528,bn
94,2023,5,52.333614,bn


In [189]:
nino_34 = sst_anomaly.query('-5 <= lat <= 5 and -170<= lon <= -120').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().rename({'normalized_sst_anomaly': 'nino_34'}, axis=1)

nino_4 = sst_anomaly.query('-5 <= lat <= 5 and lon <= -150 or -5 <= lat <= 5 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'nino_4'}, axis=1)

western_west_v = sst_anomaly.query('-15 <= lat <= 20 and 120 <= lon <= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'western_west_v'}, axis=1)

northern_west_v = sst_anomaly.query('20 <= lat <= 35 and lon <= -150 or 20 <= lat <= 35 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'northern_west_v'}, axis=1)

southern_west_v = sst_anomaly.query('-30 <= lat <= -15 and lon <= -150 or -30 <= lat <= -15 and lon >= 155').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'southern_west_v'}, axis=1)

SWIO = sst_anomaly.query('-50 <= lat <= -20 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'SWIO'}, axis=1)

IOD_west = sst_anomaly.query('-10 <= lat <= 10 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_west'}, axis=1)

IOD_east = sst_anomaly.query('-10 <= lat <= 0 and 90 <= lon <= 110').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_east'}, axis=1)

predictors = pd.concat([nino_34, nino_4, western_west_v, northern_west_v, southern_west_v, SWIO, IOD_west, IOD_east], axis=1)

In [191]:
def create_lead_month_mapping(target_season_name, target_start_month, lead_time):
    """
    Calculates the prediction month for a given target season and lead time.

    Args:
        target_season_name (str): The name of the target season (e.g., 'MAM').
        target_start_month (int): The numerical start month (1-12) of the target season.
        lead_time (int): The number of months lead time (e.g., 4).

    Returns:
        dict: A dictionary mapping the prediction month (int) to the target season name (str).
              Example: {11: 'MAM'} for a 4-month lead to March.
    """
    if not 1 <= target_start_month <= 12:
        raise ValueError("target_start_month must be between 1 and 12")
    if lead_time < 0:
        raise ValueError("lead_time cannot be negative")

    # Calculate the prediction month (1-12)
    # (target_start_month - lead_time - 1) gives the zero-based index offset
    # % 12 handles the wrap-around for negative results
    # + 1 converts back to 1-based month index
    prediction_month = (target_start_month - lead_time - 1) % 12 + 1

    return {prediction_month: target_season_name}

In [192]:
# --- Define the target season ---
season_name = 'MAM'
season_start = 3 # March is the 3rd month

season_info = {''}

# --- Generate dictionaries for leads 4 to 8 ---

# Lead time = 4 months (e.g., Nov -> Mar)
# P = (3 - 4 - 1) % 12 + 1 = (-2) % 12 + 1 = 10 + 1 = 11
four_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 4)

# Lead time = 5 months (e.g., Oct -> Mar)
# P = (3 - 5 - 1) % 12 + 1 = (-3) % 12 + 1 = 9 + 1 = 10
five_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 5)

# Lead time = 6 months (e.g., Sep -> Mar)
# P = (3 - 6 - 1) % 12 + 1 = (-4) % 12 + 1 = 8 + 1 = 9
six_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 6)

# Lead time = 7 months (e.g., Aug -> Mar)
# P = (3 - 7 - 1) % 12 + 1 = (-5) % 12 + 1 = 7 + 1 = 8
seven_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 7)

# Lead time = 8 months (e.g., Jul -> Mar)
# P = (3 - 8 - 1) % 12 + 1 = (-6) % 12 + 1 = 6 + 1 = 7
eight_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 8)

lead_maps = {}
for lead in range(4, 9): # Leads from 4 to 8 inclusive
    lead_maps[lead] = create_lead_month_mapping(season_name, season_start, lead)

In [193]:
# --- Refactored Code using a Loop ---

processed_dfs = {} # List to store the processed DataFrames for each lead time

# Loop through lead times from 4 to 8
for lead_time in range(4, 9):
    print(f"Processing lead time: {lead_time}")

    # Check if the mapping exists for the current lead time
    if lead_time not in lead_maps:
        print(f"Warning: Mapping for lead time {lead_time} not found. Skipping.")
        continue

    current_map = lead_maps[lead_time]

    # 1. Copy the base predictors DataFrame
    temp_df = predictors.copy()

    # 2. Map 'effect_season' using the correct lead time map
    temp_df['effect_season'] = temp_df['month'].map(current_map)

    year_adjustment = 0
    if lead_time >= season_start: # Check if lead time crosses start of year boundary
        year_adjustment = 1

    # Apply the adjustment to create the 'effect_year'
    # It's better to create 'effect_year' than to modify 'year' in place if 'year' is needed elsewhere
    if 'year' in temp_df.columns:
        temp_df['effect_year'] = temp_df['year'] + year_adjustment
    else:
        print(f"Warning: 'year' column not found for 'effect_year' calculation for lead {lead_time}.")

    # 3. Drop rows where mapping failed (NaN in 'effect_season') and drop 'month' column
    temp_df = temp_df.dropna(subset=['effect_season'])
    # Only drop 'month' if it exists, prevent errors
    if 'month' in temp_df.columns:
         temp_df = temp_df.drop('month', axis=1)
    else:
         print(f"Warning: 'month' column not found in temp_df for lead {lead_time} before dropping.")


    # --- Optional: Add lead time column if needed for identification later ---
    #temp_df['lead_time'] = lead_time

    # 4. Append the processed DataFrame to the list
    processed_dfs[f'lead_time_{lead_time}'] = temp_df

Processing lead time: 4
Processing lead time: 5
Processing lead time: 6
Processing lead time: 7
Processing lead time: 8


In [194]:
# Assume 'processed_dfs' is the list of DataFrames obtained from your previous loop.
# Each df in processed_dfs corresponds to lead times 4, 5, 6, 7, 8 respectively.
# Example: processed_dfs = [df_lead4, df_lead5, df_lead6, df_lead7, df_lead8]

# --- Step 1: Define Merge Keys and Lead Times ---

# !!! IMPORTANT: Verify these are the correct columns to uniquely identify rows
# !!!           These columns MUST exist in all DataFrames inside processed_dfs.
merge_keys = ['effect_season', 'effect_year'] # ADJUST AS NEEDED!


# List of lead times corresponding to the DataFrames in processed_dfs
lead_times = list(range(4, 9)) # Corresponds to leads 4, 5, 6, 7, 8

# Check if the number of dataframes matches the number of lead times
if len(processed_dfs) != len(lead_times):
    raise ValueError(f"Mismatch between number of dataframes ({len(processed_dfs)}) and lead times ({len(lead_times)})")

# --- Step 2: Rename columns in each DataFrame (excluding merge keys) ---

renamed_dfs = []
for key, df in processed_dfs.items():
    lead = key.split('_')[-1]
    suffix = f"_L{lead}"

    # Create a copy to avoid modifying the original dfs in the list if needed later
    df_renamed = df.copy().drop(['year'], axis=1)

    # Check if all merge keys exist in the current DataFrame
    missing_keys = [key for key in merge_keys if key not in df_renamed.columns]
    if missing_keys:
        raise ValueError(f"Merge key(s) {missing_keys} not found in DataFrame for lead {lead}")

    # Rename columns that are NOT in merge_keys
    cols_to_rename = {col: f"{col}{suffix}" for col in df_renamed.columns if col not in merge_keys}
    df_renamed = df_renamed.rename(columns=cols_to_rename)

    renamed_dfs.append(df_renamed)

# --- Step 3: Merge Horizontally ---

if not renamed_dfs:
    print("No dataframes to merge.")
    merged_predictors_seasonal = pd.DataFrame()
else:
    # Start with the first DataFrame
    merged_predictors_seasonal = renamed_dfs[0]

    # Iteratively merge the rest using an outer join
    for i in range(1, len(renamed_dfs)):
        try:
            merged_predictors_seasonal = pd.merge(
                merged_predictors_seasonal,
                renamed_dfs[i],
                on=merge_keys,
                how='outer' # Use 'outer' to keep all rows from all lead times
                            # Use 'inner' if you only want rows present in ALL lead times
            )
        except KeyError as e:
             print(f"\nError merging DataFrame for lead {lead_times[i]}. Missing key(s): {e}")
             print(f"Columns in left df: {merged_predictors_seasonal.columns.tolist()}")
             print(f"Columns in right df: {renamed_dfs[i].columns.tolist()}")
             # Handle error appropriately, e.g., break or continue

    # --- Alternative using reduce (more concise for many dataframes) ---
    # from functools import reduce
    # merge_func = lambda left, right: pd.merge(left, right, on=merge_keys, how='outer')
    # merged_predictors_seasonal = reduce(merge_func, renamed_dfs)

    print("\nHorizontal merge complete. Info of the merged DataFrame:")
    merged_predictors_seasonal.info()
    print("\nFirst 5 rows of merged DataFrame:")
    print(merged_predictors_seasonal.head())


Horizontal merge complete. Info of the merged DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 42 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nino_34_L4          33 non-null     float32
 1   nino_4_L4           33 non-null     float32
 2   western_west_v_L4   33 non-null     float32
 3   northern_west_v_L4  33 non-null     float32
 4   southern_west_v_L4  33 non-null     float32
 5   SWIO_L4             33 non-null     float32
 6   IOD_west_L4         33 non-null     float32
 7   IOD_east_L4         33 non-null     float32
 8   effect_season       33 non-null     object 
 9   effect_year         33 non-null     int32  
 10  nino_34_L5          33 non-null     float32
 11  nino_4_L5           33 non-null     float32
 12  western_west_v_L5   33 non-null     float32
 13  northern_west_v_L5  33 non-null     float32
 14  southern_west_v_L5  33 non-null     float32
 15  S

In [195]:
ml_data_seasonal = labeled_chirps_seasonal.merge(merged_predictors_seasonal, left_on=['year', 'season'], right_on=['effect_year', 'effect_season'], how='left').drop(['effect_year', 'effect_season'], axis=1)
ml_data_seasonal.to_csv('data/ml_data/ml_data_seasonal.csv')

In [196]:
def create_lead_month_mapping_monthly(target_start_month, lead_time):
    """
    Calculates the prediction month for a given target season and lead time.

    Args:
        target_season_name (str): The name of the target season (e.g., 'MAM').
        target_start_month (int): The numerical start month (1-12) of the target season.
        lead_time (int): The number of months lead time (e.g., 4).

    Returns:
        dict: A dictionary mapping the prediction month (int) to the target season name (str).
              Example: {11: 'MAM'} for a 4-month lead to March.
    """
    if not 1 <= target_start_month <= 12:
        raise ValueError("target_start_month must be between 1 and 12")
    if lead_time < 0:
        raise ValueError("lead_time cannot be negative")

    # Calculate the prediction month (1-12)
    # (target_start_month - lead_time - 1) gives the zero-based index offset
    # % 12 handles the wrap-around for negative results
    # + 1 converts back to 1-based month index
    prediction_month = (target_start_month - lead_time - 1) % 12 + 1
    if prediction_month >= 12:
        prediction_month_1 = prediction_month - 11
    else:
        prediction_month_1 = prediction_month + 1
    if prediction_month >= 11:
        prediction_month_2 = prediction_month - 10
    else:
        prediction_month_2 = prediction_month + 2
    target_start_month_1 = target_start_month % 12 + 1
    target_start_month_2 = target_start_month % 12 + 2

    return {prediction_month: target_start_month, prediction_month_1: target_start_month_1, prediction_month_2: target_start_month_2}

In [197]:
# --- Define the target season ---
season_start = 3  # March is the 3rd month

# --- Generate dictionaries for leads 4 to 8 ---

# Lead time = 6 months (e.g., Sep -> Mar)
# P = (3 - 6 - 1) % 12 + 1 = (-4) % 12 + 1 = 8 + 1 = 9
six_month_lead_monthly = create_lead_month_mapping_monthly(season_start, 6)

# Lead time = 7 months (e.g., Aug -> Mar)
# P = (3 - 7 - 1) % 12 + 1 = (-5) % 12 + 1 = 7 + 1 = 8
seven_month_lead_monthly = create_lead_month_mapping_monthly(season_start, 7)

# Lead time = 8 months (e.g., Jul -> Mar)
# P = (3 - 8 - 1) % 12 + 1 = (-6) % 12 + 1 = 6 + 1 = 7
eight_month_lead_monthly = create_lead_month_mapping_monthly(season_start, 8)

lead_maps = {}
for lead in range(6, 9):  # Leads from 4 to 8 inclusive
    lead_maps[lead] = create_lead_month_mapping_monthly(season_start, lead)

In [198]:
# --- Refactored Code using a Loop ---

processed_dfs = {}  # List to store the processed DataFrames for each lead time

# Loop through lead times from 4 to 8
for lead_time in range(6, 9):
    print(f"Processing lead time: {lead_time}")

    # Check if the mapping exists for the current lead time
    if lead_time not in lead_maps:
        print(f"Warning: Mapping for lead time {lead_time} not found. Skipping.")
        continue

    current_map = lead_maps[lead_time]

    # 1. Copy the base predictors DataFrame
    temp_df = predictors.copy()

    # 2. Map 'effect_season' using the correct lead time map
    temp_df['effect_month'] = temp_df['month'].map(current_map)

    year_adjustment = 0
    if lead_time >= season_start:  # Check if lead time crosses start of year boundary
        year_adjustment = 1

    # Apply the adjustment to create the 'effect_year'
    # It's better to create 'effect_year' than to modify 'year' in place if 'year' is needed elsewhere
    if 'year' in temp_df.columns:
        temp_df['effect_year'] = temp_df['year'] + year_adjustment
    else:
        print(f"Warning: 'year' column not found for 'effect_year' calculation for lead {lead_time}.")

    # 3. Drop rows where mapping failed (NaN in 'effect_season') and drop 'month' column
    temp_df = temp_df.dropna(subset=['effect_month'])
    # Only drop 'month' if it exists, prevent errors
    if 'month' in temp_df.columns:
        temp_df = temp_df.drop('month', axis=1)
    else:
        print(f"Warning: 'month' column not found in temp_df for lead {lead_time} before dropping.")

    # --- Optional: Add lead time column if needed for identification later ---
    #temp_df['lead_time'] = lead_time

    # 4. Append the processed DataFrame to the list
    processed_dfs[f'lead_time_{lead_time}'] = temp_df

Processing lead time: 6
Processing lead time: 7
Processing lead time: 8


In [199]:
# Assume 'processed_dfs' is the list of DataFrames obtained from your previous loop.
# Each df in processed_dfs corresponds to lead times 4, 5, 6, 7, 8 respectively.
# Example: processed_dfs = [df_lead4, df_lead5, df_lead6, df_lead7, df_lead8]

# --- Step 1: Define Merge Keys and Lead Times ---

# !!! IMPORTANT: Verify these are the correct columns to uniquely identify rows
# !!!           These columns MUST exist in all DataFrames inside processed_dfs.
merge_keys = ['effect_month', 'effect_year']  # ADJUST AS NEEDED!

# List of lead times corresponding to the DataFrames in processed_dfs
lead_times = list(range(6, 9))  # Corresponds to leads 4, 5, 6, 7, 8

# Check if the number of dataframes matches the number of lead times
if len(processed_dfs) != len(lead_times):
    raise ValueError(f"Mismatch between number of dataframes ({len(processed_dfs)}) and lead times ({len(lead_times)})")

# --- Step 2: Rename columns in each DataFrame (excluding merge keys) ---

renamed_dfs = []
for key, df in processed_dfs.items():
    lead = key.split('_')[-1]
    suffix = f"_L{lead}"

    # Create a copy to avoid modifying the original dfs in the list if needed later
    df_renamed = df.copy().drop(['year'], axis=1)

    # Check if all merge keys exist in the current DataFrame
    missing_keys = [key for key in merge_keys if key not in df_renamed.columns]
    if missing_keys:
        raise ValueError(f"Merge key(s) {missing_keys} not found in DataFrame for lead {lead}")

    # Rename columns that are NOT in merge_keys
    cols_to_rename = {col: f"{col}{suffix}" for col in df_renamed.columns if col not in merge_keys}
    df_renamed = df_renamed.rename(columns=cols_to_rename)

    renamed_dfs.append(df_renamed)

# --- Step 3: Merge Horizontally ---

if not renamed_dfs:
    print("No dataframes to merge.")
    merged_predictors_monthly = pd.DataFrame()
else:
    # Start with the first DataFrame
    merged_predictors_monthly = renamed_dfs[0]

    # Iteratively merge the rest using an outer join
    for i in range(1, len(renamed_dfs)):
        try:
            merged_predictors_monthly = pd.merge(
                merged_predictors_monthly,
                renamed_dfs[i],
                on=merge_keys,
                how='outer'  # Use 'outer' to keep all rows from all lead times
                # Use 'inner' if you only want rows present in ALL lead times
            )
        except KeyError as e:
            print(f"\nError merging DataFrame for lead {lead_times[i]}. Missing key(s): {e}")
            print(f"Columns in left df: {merged_predictors_monthly.columns.tolist()}")
            print(f"Columns in right df: {renamed_dfs[i].columns.tolist()}")
            # Handle error appropriately, e.g., break or continue

    # --- Alternative using reduce (more concise for many dataframes) ---
    # from functools import reduce
    # merge_func = lambda left, right: pd.merge(left, right, on=merge_keys, how='outer')
    # merged_predictors_monthly = reduce(merge_func, renamed_dfs)

    print("\nHorizontal merge complete. Info of the merged DataFrame:")
    merged_predictors_monthly.info()
    print("\nFirst 5 rows of merged DataFrame:")
    print(merged_predictors_monthly.head())


Horizontal merge complete. Info of the merged DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nino_34_L6          99 non-null     float32
 1   nino_4_L6           99 non-null     float32
 2   western_west_v_L6   99 non-null     float32
 3   northern_west_v_L6  99 non-null     float32
 4   southern_west_v_L6  99 non-null     float32
 5   SWIO_L6             99 non-null     float32
 6   IOD_west_L6         99 non-null     float32
 7   IOD_east_L6         99 non-null     float32
 8   effect_month        99 non-null     float64
 9   effect_year         99 non-null     int32  
 10  nino_34_L7          99 non-null     float32
 11  nino_4_L7           99 non-null     float32
 12  western_west_v_L7   99 non-null     float32
 13  northern_west_v_L7  99 non-null     float32
 14  southern_west_v_L7  99 non-null     float32
 15  S

In [201]:
ml_data_monthly = labeled_chirps_monthly.merge(merged_predictors_monthly, left_on=['year', 'month'], right_on=['effect_year', 'effect_month'], how='left').drop(['effect_year', 'effect_month'], axis=1)
ml_data_monthly.to_csv('data/ml_data/eea_mam_ml_data_monthly.csv')

In [205]:
ml_data_seasonal_tercile = labeled_chirps_monthly_seasonal_tercile.merge(merged_predictors_monthly, left_on=['year', 'month'], right_on=['effect_year', 'effect_month'], how='left').drop(['effect_year', 'effect_month'], axis=1)
ml_data_seasonal_tercile.to_csv('data/ml_data/eea_mam_ml_data_monthly_seasonal_tercile.csv')